# Multi-Agent Stock Research Desk
### Week 4 Capstone Project

**Problem statement:** A single LLM agent asked to "analyse this stock" tends to blend technical, fundamental, and risk reasoning into one shallow answer. This system fixes that with **three independent analyst agents** that each see only their own slice of data, plus an **aggregator agent** that reconciles their (possibly conflicting) verdicts into one final brief.

**Orchestration pattern: Parallel + Aggregator**

**Agents & division of responsibility:**

| Agent | Sees only | Model |
|---|---|---|
| Technical Analyst | RSI(14), SMA20/SMA50, recent volatility | Groq — llama-3.3-70b-versatile |
| Fundamental Analyst | P/E, market cap, sector, margins, growth | Groq — llama-3.3-70b-versatile |
| Risk Analyst | volatility, max drawdown, beta, Sharpe | Groq — llama-3.3-70b-versatile |
| Aggregator | the 3 finished verdicts (not raw data) | Gemini 2.5 Flash |


## 1. Install dependencies

In [1]:
%pip install -q langgraph langchain-groq langchain-google-genai yfinance numpy pandas

Note: you may need to restart the kernel to use updated packages.


## 2. API Keys

In [2]:
import os
from getpass import getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Enter your Gemini API key: ")

print("Groq key set:", bool(os.environ.get("GROQ_API_KEY")))
print("Gemini key set:", bool(os.environ.get("GOOGLE_API_KEY")))

Groq key set: True
Gemini key set: True


## 3. Config

In [3]:
TICKER = "RELIANCE.NS"
PERIOD = "1y"          
BENCHMARK = "^NSEI"   

GROQ_MODEL_NAME = "llama-3.3-70b-versatile"
GEMINI_MODEL_NAME = "gemini-2.5-flash"

GROQ_RPM_LIMIT, GROQ_RPD_LIMIT = 30, 1000
GEMINI_RPM_LIMIT, GEMINI_RPD_LIMIT = 10, 250

## 4. Imports

In [4]:
import time
from collections import deque
from typing import TypedDict, Optional

import numpy as np
import pandas as pd
import yfinance as yf

from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage

## 5. Groq for analysts, Gemini for aggregator

In [5]:
sub_agent_llm = ChatGroq(model=GROQ_MODEL_NAME, temperature=0.3, max_tokens=500)
aggregator_llm = ChatGoogleGenerativeAI(
    model=GEMINI_MODEL_NAME,
    temperature=0.3,
    max_tokens=2048,
    thinking_budget=0, 
)
print(f"Sub-agents -> Groq/{GROQ_MODEL_NAME}")
print(f"Aggregator -> Gemini/{GEMINI_MODEL_NAME}")

Sub-agents -> Groq/llama-3.3-70b-versatile
Aggregator -> Gemini/gemini-2.5-flash


## 6. Rate Limit Monitor

In [6]:
class RateLimitMonitor:
    def __init__(self, name: str, rpm_limit: int, rpd_limit: int):
        self.name = name
        self.rpm_limit = rpm_limit
        self.rpd_limit = rpd_limit
        self._minute_window: deque = deque()
        self._day_count = 0
        self._day_start = time.time()

    def _reset_day_if_needed(self):
        if time.time() - self._day_start > 86400:
            self._day_count = 0
            self._day_start = time.time()

    def _prune_window(self):
        now = time.time()
        while self._minute_window and now - self._minute_window[0] > 60:
            self._minute_window.popleft()

    def before_call(self):
        '''Call this right before making an API request.'''
        self._reset_day_if_needed()
        self._prune_window()

        if self._day_count >= self.rpd_limit:
            raise RuntimeError(
                f"[{self.name}] Daily request limit reached "
                f"({self._day_count}/{self.rpd_limit} RPD). "
                f"Wait for the daily reset or raise the RPD limit if you have a higher tier."
            )

        if len(self._minute_window) >= self.rpm_limit:
            oldest = self._minute_window[0]
            wait_time = 60 - (time.time() - oldest) + 0.25
            if wait_time > 0:
                print(f"[RateLimitMonitor:{self.name}] Hit {self.rpm_limit} RPM cap, "
                      f"waiting {wait_time:.1f}s before next call...")
                time.sleep(wait_time)
            self._prune_window()

    def record_call(self):
        '''Call this right after a successful API request.'''
        self._minute_window.append(time.time())
        self._day_count += 1

    def status(self) -> str:
        return (f"[{self.name}] {len(self._minute_window)}/{self.rpm_limit} RPM used | "
                f"{self._day_count}/{self.rpd_limit} RPD used")


groq_monitor = RateLimitMonitor("Groq", GROQ_RPM_LIMIT, GROQ_RPD_LIMIT)
gemini_monitor = RateLimitMonitor("Gemini", GEMINI_RPM_LIMIT, GEMINI_RPD_LIMIT)


def call_llm(llm, monitor: RateLimitMonitor, messages):
    '''Rate-limit-aware wrapper around llm.invoke(). Every LLM call in this
    notebook goes through here.'''
    monitor.before_call()
    response = llm.invoke(messages)
    monitor.record_call()
    print(f"    {monitor.status()}")
    return response

## 7. Shared state schema

In [7]:
class ResearchState(TypedDict, total=False):
    ticker: str
    period: str
    benchmark: str

    price_df: pd.DataFrame
    bench_df: pd.DataFrame
    info: dict

    technical_report: str
    fundamental_report: str
    risk_report: str
    final_report: str

## 8. Node

In [8]:
def fetch_data(state: ResearchState) -> ResearchState:
    ticker = state["ticker"]
    period = state.get("period", "6mo")
    benchmark = state.get("benchmark", "^NSEI")

    print("=" * 70)
    print("STEP 1 — DATA COLLECTION")
    print("=" * 70)
    print(f"[fetch_data] Pulling {period} of price history for {ticker} "
          f"and benchmark {benchmark} ...")

    price_df = yf.Ticker(ticker).history(period=period)
    if price_df.empty:
        raise ValueError(
            f"No price data returned for '{ticker}'. Check the symbol "
            f"(Indian tickers need a suffix, e.g. 'RELIANCE.NS')."
        )

    bench_df = yf.Ticker(benchmark).history(period=period)
    info = yf.Ticker(ticker).info or {}

    print(f"[fetch_data] Got {len(price_df)} trading days for {ticker} "
          f"({price_df.index[0].date()} -> {price_df.index[-1].date()})")
    print(f"[fetch_data] Got {len(bench_df)} trading days for benchmark {benchmark}")
    print(f"[fetch_data] Latest close -> {ticker}: {price_df['Close'].iloc[-1]:.2f} | "
          f"{benchmark}: {bench_df['Close'].iloc[-1]:.2f}")
    print(f"[fetch_data] Company info -> sector: {info.get('sector', 'N/A')}, "
          f"industry: {info.get('industry', 'N/A')}")
    print("\n[fetch_data] Handing off to 3 analyst agents in parallel...\n")

    return {"price_df": price_df, "bench_df": bench_df, "info": info}

## 9. Indicator helper function

In [9]:
def compute_rsi(close: pd.Series, window: int = 14) -> float:
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(window).mean()
    avg_loss = loss.rolling(window).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    return float(rsi.iloc[-1]) if not rsi.empty and not np.isnan(rsi.iloc[-1]) else 50.0


def compute_technical_metrics(price_df: pd.DataFrame) -> dict:
    close = price_df["Close"]
    sma20 = close.rolling(20).mean().iloc[-1]
    sma50 = close.rolling(50).mean().iloc[-1] if len(close) >= 50 else np.nan
    last_price = close.iloc[-1]
    rsi = compute_rsi(close)
    daily_ret = close.pct_change().dropna()
    recent_vol_annualized = daily_ret.tail(20).std() * np.sqrt(252) * 100
    pct_change_period = (last_price / close.iloc[0] - 1) * 100

    return {
        "last_price": round(float(last_price), 2),
        "sma20": round(float(sma20), 2) if not np.isnan(sma20) else None,
        "sma50": round(float(sma50), 2) if not np.isnan(sma50) else None,
        "rsi_14": round(rsi, 1),
        "recent_volatility_annualized_pct": round(float(recent_vol_annualized), 1),
        "period_return_pct": round(float(pct_change_period), 1),
    }


def compute_risk_metrics(price_df: pd.DataFrame, bench_df: pd.DataFrame) -> dict:
    stock_ret = price_df["Close"].pct_change().dropna()
    bench_ret = bench_df["Close"].pct_change().dropna()

    aligned = pd.concat([stock_ret, bench_ret], axis=1, join="inner")
    aligned.columns = ["stock", "bench"]

    ann_vol = stock_ret.std() * np.sqrt(252) * 100

    cum = (1 + stock_ret).cumprod()
    running_max = cum.cummax()
    drawdown = (cum / running_max - 1)
    max_drawdown = drawdown.min() * 100

    if len(aligned) > 2 and aligned["bench"].var() > 0:
        beta = aligned["stock"].cov(aligned["bench"]) / aligned["bench"].var()
    else:
        beta = float("nan")

    mean_daily = stock_ret.mean()
    std_daily = stock_ret.std()
    sharpe_approx = (mean_daily / std_daily) * np.sqrt(252) if std_daily > 0 else float("nan")

    return {
        "annualized_volatility_pct": round(float(ann_vol), 1),
        "max_drawdown_pct": round(float(max_drawdown), 1),
        "beta_vs_benchmark": round(float(beta), 2) if not np.isnan(beta) else None,
        "sharpe_ratio_approx": round(float(sharpe_approx), 2) if not np.isnan(sharpe_approx) else None,
    }


def extract_fundamentals(info: dict) -> dict:
    keys_wanted = {
        "sector": "sector",
        "industry": "industry",
        "marketCap": "market_cap",
        "trailingPE": "trailing_pe",
        "forwardPE": "forward_pe",
        "profitMargins": "profit_margins",
        "revenueGrowth": "revenue_growth",
        "dividendYield": "dividend_yield",
        "recommendationKey": "analyst_recommendation",
    }
    return {out_key: info.get(src_key, "N/A") for src_key, out_key in keys_wanted.items()}

## 10. Agent — Technical Analyst (parallel branch #1, Groq)

In [10]:
def technical_analyst(state: ResearchState) -> ResearchState:
    print("[Technical Analyst] Reading price action & indicators for "
          f"{state['ticker']}...")
    t0 = time.time()
    metrics = compute_technical_metrics(state["price_df"])
    print(f"[Technical Analyst] Computed metrics: {metrics}")

    system = SystemMessage(content=(
        "You are a technical analyst on an equity research desk. You ONLY "
        "look at price action and technical indicators -- you do not know "
        "anything about the company's fundamentals or risk metrics. Give a "
        "7-8 sentence, detailed read, ending with a clear stance label: "
        "Bullish, Bearish, or Neutral."
    ))
    human = HumanMessage(content=f"Ticker: {state['ticker']}\nTechnical data: {metrics}")

    print("[Technical Analyst] Consulting Groq (llama-3.3-70b-versatile)...")
    response = call_llm(sub_agent_llm, groq_monitor, [system, human])
    elapsed = time.time() - t0
    print(f"[Technical Analyst] Done in {elapsed:.1f}s. Verdict:")
    print(f"    {response.content}\n")
    return {"technical_report": response.content}

## 11. Agent — Fundamental Analyst (parallel branch #2, Groq)

In [11]:
def fundamental_analyst(state: ResearchState) -> ResearchState:
    print("[Fundamental Analyst] Reviewing company fundamentals for "
          f"{state['ticker']}...")
    t0 = time.time()
    fundamentals = extract_fundamentals(state["info"])
    print(f"[Fundamental Analyst] Extracted fundamentals: {fundamentals}")

    system = SystemMessage(content=(
        "You are a fundamentals analyst on an equity research desk. You ONLY "
        "look at company fundamentals (valuation, margins, growth, sector) -- "
        "you do not see the price chart or risk metrics. Give a 7-8 "
        "sentence detailed read, ending with a clear stance label: Bullish, Bearish, "
        "or Neutral."
    ))
    human = HumanMessage(content=f"Ticker: {state['ticker']}\nFundamental data: {fundamentals}")

    print("[Fundamental Analyst] Consulting Groq (llama-3.3-70b-versatile)...")
    response = call_llm(sub_agent_llm, groq_monitor, [system, human])
    elapsed = time.time() - t0
    print(f"[Fundamental Analyst] Done in {elapsed:.1f}s. Verdict:")
    print(f"    {response.content}\n")
    return {"fundamental_report": response.content}

## 12. Agent — Risk Analyst (parallel branch #3, Groq)

In [12]:
def risk_analyst(state: ResearchState) -> ResearchState:
    print("[Risk Analyst] Assessing volatility, drawdown & beta for "
          f"{state['ticker']}...")
    t0 = time.time()
    risk = compute_risk_metrics(state["price_df"], state["bench_df"])
    print(f"[Risk Analyst] Computed risk metrics: {risk}")

    system = SystemMessage(content=(
        "You are a risk officer on an equity research desk. You ONLY look at "
        "volatility, drawdown, beta, and risk-adjusted return metrics -- you "
        "do not see the price chart shape or company fundamentals. Give a "
        "7-8 sentence detailed read, ending with a clear risk label: Low Risk, "
        "Moderate Risk, or High Risk."
    ))
    human = HumanMessage(content=f"Ticker: {state['ticker']}\nRisk data: {risk}")

    print("[Risk Analyst] Consulting Groq (llama-3.3-70b-versatile)...")
    response = call_llm(sub_agent_llm, groq_monitor, [system, human])
    elapsed = time.time() - t0
    print(f"[Risk Analyst] Done in {elapsed:.1f}s. Verdict:")
    print(f"    {response.content}\n")
    return {"risk_report": response.content}

## 13. Agent — Aggregator (Gemini 2.5 Flash)

In [13]:
def aggregator(state: ResearchState) -> ResearchState:
    print("=" * 70)
    print("STEP 3 — SYNTHESIS")
    print("=" * 70)
    print("[Aggregator / Portfolio Manager] All 3 analyst notes received. "
          "Reconciling independent opinions...")
    t0 = time.time()

    system = SystemMessage(content=(
        "You are the portfolio manager on a research desk. Three analysts "
        "have independently filed their notes below, each seeing only their "
        "own slice of data. Your job:\n"
        "1. Summarize where the three analysts AGREE.\n"
        "2. Explicitly flag where they DISAGREE and why that might be.\n"
        "3. Give one final overall stance (Bullish / Bearish / Neutral) with "
        "a confidence level (Low / Medium / High).\n"
        "Keep it around 400 words, structured with short headers."
    ))
    human = HumanMessage(content=(
        f"Ticker: {state['ticker']}\n\n"
        f"--- Technical Analyst ---\n{state['technical_report']}\n\n"
        f"--- Fundamental Analyst ---\n{state['fundamental_report']}\n\n"
        f"--- Risk Analyst ---\n{state['risk_report']}\n"
    ))

    print("[Aggregator] Consulting Gemini 2.5 Flash for final synthesis...")
    response = call_llm(aggregator_llm, gemini_monitor, [system, human])
    elapsed = time.time() - t0
    print(f"[Aggregator] Final brief ready in {elapsed:.1f}s.\n")
    return {"final_report": response.content}

## 14. This is the Parallel + Aggregator pattern

In [14]:
def build_graph():
    graph = StateGraph(ResearchState)

    graph.add_node("fetch_data", fetch_data)
    graph.add_node("technical_analyst", technical_analyst)
    graph.add_node("fundamental_analyst", fundamental_analyst)
    graph.add_node("risk_analyst", risk_analyst)
    graph.add_node("aggregator", aggregator)

    graph.add_edge(START, "fetch_data")

    graph.add_edge("fetch_data", "technical_analyst")
    graph.add_edge("fetch_data", "fundamental_analyst")
    graph.add_edge("fetch_data", "risk_analyst")

    graph.add_edge("technical_analyst", "aggregator")
    graph.add_edge("fundamental_analyst", "aggregator")
    graph.add_edge("risk_analyst", "aggregator")

    graph.add_edge("aggregator", END)

    return graph.compile()

app = build_graph()
print("Graph compiled successfully.")

Graph compiled successfully.


## 15. Researches the ticker set

In [15]:
print("#" * 70)
print(f"#  MULTI-AGENT STOCK RESEARCH DESK -- Researching {TICKER}")
print("#" * 70)
print()

initial_state: ResearchState = {
    "ticker": TICKER,
    "period": PERIOD,
    "benchmark": BENCHMARK,
}

_run_start = time.time()
result = app.invoke(initial_state)
_run_elapsed = time.time() - _run_start

print("=" * 70)
print(f"ALL AGENTS FINISHED -- total run time: {_run_elapsed:.1f}s")
print("=" * 70)

######################################################################
#  MULTI-AGENT STOCK RESEARCH DESK -- Researching RELIANCE.NS
######################################################################

STEP 1 — DATA COLLECTION
[fetch_data] Pulling 1y of price history for RELIANCE.NS and benchmark ^NSEI ...
[fetch_data] Got 249 trading days for RELIANCE.NS (2025-07-07 -> 2026-07-06)
[fetch_data] Got 245 trading days for benchmark ^NSEI
[fetch_data] Latest close -> RELIANCE.NS: 1318.70 | ^NSEI: 24405.35
[fetch_data] Company info -> sector: Energy, industry: Oil & Gas Refining & Marketing

[fetch_data] Handing off to 3 analyst agents in parallel...

[Fundamental Analyst] Reviewing company fundamentals for RELIANCE.NS...
[Fundamental Analyst] Extracted fundamentals: {'sector': 'Energy', 'industry': 'Oil & Gas Refining & Marketing', 'market_cap': 17845270347776, 'trailing_pe': 22.081379, 'forward_pe': 18.366795, 'profit_margins': 0.0764, 'revenue_growth': 0.125, 'dividend_yield': 0.46, '

## 16. Final research brief

In [16]:
print("=" * 70)
print(f"  FINAL RESEARCH BRIEF -- {TICKER}")
print("=" * 70)
print(result["final_report"])

  FINAL RESEARCH BRIEF -- RELIANCE.NS
## Portfolio Manager's Summary: RELIANCE.NS

### Areas of Agreement

All three analysts implicitly or explicitly acknowledge the **moderate volatility** of RELIANCE.NS. The Technical Analyst notes "moderate volatility of 16.2%", while the Risk Analyst states "annualized volatility of 20.1%", both falling within a similar range. There's also a shared understanding that the stock is a **large-cap entity** (Fundamental Analyst explicitly states this, while the others' metrics like market cap and widespread analysis imply it).

### Areas of Disagreement

**1. Overall Stance:**
*   **Technical Analyst:** Bearish, citing the stock trading below its 50-day SMA and a negative period return.
*   **Fundamental Analyst:** Bullish, based on strong revenue growth, premium valuation (implying future growth), and a "strong buy" analyst recommendation.
*   **Risk Analyst:** Does not provide a direct bullish/bearish stance but labels the stock as "High Risk," sugge

## 17. Individual analyst notes (for transparency / grading)

In [17]:
print("-" * 70)
print("TECHNICAL ANALYST")
print("-" * 70)
print(result["technical_report"])

print("\n" + "-" * 70)
print("FUNDAMENTAL ANALYST")
print("-" * 70)
print(result["fundamental_report"])

print("\n" + "-" * 70)
print("RISK ANALYST")
print("-" * 70)
print(result["risk_report"])

----------------------------------------------------------------------
TECHNICAL ANALYST
----------------------------------------------------------------------
The price action of RELIANCE.NS is currently trading below its 50-day simple moving average (SMA) of 1338.46, which may indicate a bearish trend. However, the stock is slightly above its 20-day SMA of 1305.25, suggesting some short-term support. The Relative Strength Index (RSI) of 46.0 is neutral, indicating that the stock is not oversold or overbought. The recent volatility of 16.2% is moderate, which could lead to further price swings. The period return of -13.7% is a concern, as it indicates a decline in the stock's value. Considering these factors, the overall technical outlook is leaning towards a bearish stance due to the stock's inability to surpass its 50-day SMA and the negative period return. The neutral RSI and moderate volatility do not provide a strong enough signal to override the bearish indicators. Overall stanc